# 1D Double-Well Potential: Numerical Experiments

Implements Section 3.2 of the thesis. Covers verification of the DL-FP method, initialization sensitivity analysis, effect of optimizer configuration, and transfer learning.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

sys.path.append(os.path.abspath('..'))

# Import our custom modules
from src.problems.double_well import DoubleWellProblem
from src.models import FPNet
from src.training import train_model
from src.utils import acc_L2
from src.losses import fp_loss

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3.2.1 Verification on Standard Configurations

Trains the standard PINN (no normalization) and the DL-FP method for $\alpha \in \{0.3,\,0.5,\,0.7\}$. Reproduces Figure 3.1.

In [ ]:
# 3.2.1 Figure 3.1 Verification on standard configurations
problems = {
    'alpha=0.3': DoubleWellProblem(alpha=0.3, beta=0.5, sigma=0.5),
    'alpha=0.5': DoubleWellProblem(alpha=0.5, beta=0.5, sigma=0.5),
    'alpha=0.7': DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5),
}

x_eval = np.linspace(-2.2, 2.2, 441)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (name, problem) in enumerate(problems.items()):

    print("  -> Baseline PINN (use_normalization=False)")
    result_baseline = train_model(
        problem,
        model_config={'output_transform': 'softplus'},
        train_config={'epochs': 30000, 'lr': 1e-3},
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        use_normalization=False,
        seed=42,
        print_every=15000,
        device=device
    )

    print("  -> DL-FP (use_normalization=True)")
    result_dlfp = train_model(
        problem,
        model_config={'output_transform': 'softplus'},
        train_config={'epochs': 30000, 'lr': 1e-3},
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        use_normalization=True,
        seed=42,
        print_every=15000,
        device=device
    )

    model_baseline = result_baseline['model']
    model_dlfp = result_dlfp['model']

    model_baseline.eval()
    model_dlfp.eval()

    x_t = torch.tensor(x_eval, dtype=torch.float32).view(-1, 1).to(device)

    with torch.no_grad():
        p_pred_baseline = model_baseline(x_t).cpu().numpy().flatten()
        p_pred_dlfp = model_dlfp(x_t).cpu().numpy().flatten()

    p_exact = problem.exact_solution(x_eval)

    acc_baseline = acc_L2(p_pred_baseline, p_exact)
    acc_dlfp = acc_L2(p_pred_dlfp, p_exact)

    axes[i].plot(x_eval, p_exact, 'k-', lw=2, alpha=0.6, label='Exact')
    axes[i].plot(x_eval, p_pred_baseline, 'b--', lw=1.5, label=f'PINN (acc={acc_baseline:.4f})')
    axes[i].plot(x_eval, p_pred_dlfp, 'r--', lw=1.5, label=f'DL-FP (acc={acc_dlfp:.4f})')

    axes[i].set_title(name)
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel('x')

plt.tight_layout()
plt.savefig('step1_baseline_vs_dlfp.png', dpi=100)
plt.show()


## 3.2.2 Initialization Sensitivity Analysis

Evaluates DL-FP across 16 independent seeds for $\alpha = 0.7$. Reproduces Figure 3.2.

In [ ]:

problem = DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5)
x_eval = np.linspace(-2.2, 2.2, 441)
p_exact = problem.exact_solution(x_eval)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_once(seed, epochs=30000):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = FPNet(input_dim=1, hidden_layers=4, neurons=20,
                  output_transform='softplus').to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    x_grid = torch.linspace(-2.2, 2.2, 441,
                            dtype=torch.float32, device=device).view(-1, 1)
    x_grid.requires_grad_(True)
    x_boundary = torch.tensor([[-2.2], [2.2]],
                              dtype=torch.float32, device=device)
    dx = 4.4 / 440

    for epoch in range(epochs):
        optimizer.zero_grad()
        losses = fp_loss(
            model=model, x=x_grid, x_boundary=x_boundary,
            mu_fn=problem.mu, D_fn=problem.D, dx=dx,
            use_normalization=True, a1=1.0, a2=1.0, a3=1.0,
        )
        losses['total'].backward()
        optimizer.step()

    model.eval()
    x_t = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)
    with torch.no_grad():
        p_pred = model(x_t).cpu().numpy().flatten()

    final_loss_pde = losses['pde'].item()
    acc = acc_L2(p_pred, p_exact)

    # Safe ratio calculation
    mid = len(x_eval) // 2
    lm = p_pred[:mid].max()
    rm = p_pred[mid:].max()
    max_val = max(lm, rm)
    ratio = min(lm, rm) / max_val if max_val > 1e-6 else 0.0

    return {
        'model': model, 'p': p_pred,
        'loss_pde': final_loss_pde,
        'acc': acc, 'ratio': ratio, 'seed': seed,
    }

# Test 1: 16 seeds, 30000 epochs each
seeds_16 = list(range(16))
results_16 = []

print("Testing 16 seeds (30000 epochs each)...")
for seed in seeds_16:
    r = train_once(seed=seed, epochs=30000)
    results_16.append(r)
    status = '✓ TWO' if r['ratio'] > 0.3 else '✗ ONE'
    print(f"  Seed={seed:>3}: acc={r['acc']:.4f}, "
          f"loss_pde={r['loss_pde']:.3e}, ratio={r['ratio']:.3f}  {status}")

# Statistics
two_peaks = [r for r in results_16 if r['ratio'] > 0.3]
one_peak  = [r for r in results_16 if r['ratio'] <= 0.3]

if two_peaks:
    print(f"\nTWO PEAKS:  {len(two_peaks)}/16 "
          f"(Acc: min={min(r['acc'] for r in two_peaks):.4f}, "
          f"max={max(r['acc'] for r in two_peaks):.4f})")
else:
    print("\nTWO PEAKS:  0/16")

print(f"ONE PEAK:   {len(one_peak)}/16")

# Test 2: Multi-restart strategy — picking the best of N runs
print("\n" + "="*50)
print("Multi-restart: Best of N runs (evaluated by lowest loss_pde)")
for n_restarts in [3, 5, 10, 16]:
    candidates = results_16[:n_restarts]
    best = min(candidates, key=lambda r: r['loss_pde'])
    print(f"  Best of {n_restarts:<2}: Seed={best['seed']:>2}, "
          f"Acc={best['acc']:.4f}, Loss PDE={best['loss_pde']:.3e}, "
          f"Ratio={best['ratio']:.3f}")

# Plot: All 16 results
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()
for i, r in enumerate(results_16):
    status = '✓' if r['ratio'] > 0.3 else '✗'
    axes[i].plot(x_eval, p_exact, 'k-', lw=1.5, alpha=0.5)
    axes[i].plot(x_eval, r['p'], 'r--', lw=1.5)
    axes[i].set_title(f"Seed={r['seed']} {status}\nAcc={r['acc']:.4f}", fontsize=11)
    axes[i].set_xlabel('x', fontsize=9)

plt.suptitle('Alpha=0.7 : 16 Random Seeds Test', fontsize=16)
plt.tight_layout()
plt.savefig('step8_multirun.png', dpi=100)
plt.show()

### Symmetry Breaking: Early-Epoch Zoom

Tracks left/right peak heights at 500-epoch intervals between epochs 2000 and 10 000, pinpointing the window where the two seeds diverge. Complements Figure 3.3 (full 30 000-epoch trajectory).

In [ ]:
# Early-epoch dynamics: locating the symmetry-breaking window (epochs 2000–10000)
problem_snap = DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5)
x_eval_snap  = np.linspace(-2.2, 2.2, 441)
device_snap  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_with_snapshots(seed, snapshot_epochs):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device_snap.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = FPNet(input_dim=1, hidden_layers=4, neurons=20,
                  output_transform='softplus').to(device_snap)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10000, gamma=0.1)

    x_grid = torch.linspace(-2.2, 2.2, 441,
                            dtype=torch.float32, device=device_snap).view(-1, 1)
    x_grid.requires_grad_(True)
    x_boundary = torch.tensor([[-2.2], [2.2]],
                               dtype=torch.float32, device=device_snap)
    dx = 4.4 / 440

    snapshots = {}
    for epoch in range(max(snapshot_epochs) + 1):
        optimizer.zero_grad()
        losses = fp_loss(
            model=model, x=x_grid, x_boundary=x_boundary,
            mu_fn=problem_snap.mu, D_fn=problem_snap.D, dx=dx,
            use_normalization=True, a1=1.0, a2=1.0, a3=1.0,
        )
        losses['total'].backward()
        optimizer.step()
        scheduler.step()

        if epoch in snapshot_epochs:
            model.eval()
            x_t = torch.tensor(x_eval_snap, dtype=torch.float32,
                                device=device_snap).view(-1, 1)
            with torch.no_grad():
                p_pred = model(x_t).cpu().numpy().flatten()
            mid = len(x_eval_snap) // 2
            snapshots[epoch] = {
                'p':         p_pred.copy(),
                'left_max':  p_pred[:mid].max(),
                'right_max': p_pred[mid:].max(),
                'loss_pde':  losses['pde'].item(),
            }
            model.train()
    return snapshots

snapshot_epochs = list(range(2000, 10001, 500))

print("Training seed=0 (success)...")
snaps_0 = train_with_snapshots(seed=0, snapshot_epochs=snapshot_epochs)
print("Training seed=1 (failure)...")
snaps_1 = train_with_snapshots(seed=1, snapshot_epochs=snapshot_epochs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (seed, snaps) in zip(axes, [(0, snaps_0), (1, snaps_1)]):
    epochs_plot = list(snaps.keys())
    left  = [snaps[e]['left_max'] for e in epochs_plot]
    right = [snaps[e]['right_max'] for e in epochs_plot]
    ratio = []
    for l, r in zip(left, right):
        max_val = max(l, r)
        ratio.append(min(l, r) / max_val if max_val > 1e-6 else 0.0)

    ax.plot(epochs_plot, left,  'b-o', ms=4, label='Left peak max')
    ax.plot(epochs_plot, right, 'r-o', ms=4, label='Right peak max')
    ax.plot(epochs_plot, ratio, 'k--s', ms=4, label='Symmetry ratio (1=perfect)')
    ax.axhline(0.3, color='gray', ls=':', lw=1, label='Threshold=0.3')
    ax.set_xlabel('Epoch')
    ax.set_title(f'Seed = {seed}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Where Symmetry Breaking Occurs', fontsize=14)
plt.tight_layout()
plt.savefig('symmetry_breaking_early.png', dpi=100)
plt.show()

### Training Dynamics: Symmetry Breaking

Tracks peak heights, symmetry ratio, and PDE loss over 30 000 epochs for a successful (seed 0) and a failed run (seed 1). Reproduces Figure 3.3.

In [ ]:

problem = DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5)
x_eval = np.linspace(-2.2, 2.2, 441)
p_exact = problem.exact_solution(x_eval)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_full(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = FPNet(input_dim=1, hidden_layers=4, neurons=20,
                  output_transform='softplus').to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10000, gamma=0.1)

    x_grid = torch.linspace(-2.2, 2.2, 441,
                            dtype=torch.float32, device=device).view(-1, 1)
    x_grid.requires_grad_(True)
    x_boundary = torch.tensor([[-2.2], [2.2]],
                              dtype=torch.float32, device=device)
    dx = 4.4 / 440

    # Record each epoch for a precise plot
    history = {
        'epoch': [], 'left_max': [], 'right_max': [],
        'ratio': [], 'loss_pde': [], 'lr': []
    }

    snapshot_epochs = set(range(0, 30001, 200))
    snapshots = {}

    for epoch in range(30001):
        optimizer.zero_grad()
        losses = fp_loss(
            model=model, x=x_grid, x_boundary=x_boundary,
            mu_fn=problem.mu, D_fn=problem.D, dx=dx,
            use_normalization=True, a1=1.0, a2=1.0, a3=1.0,
        )
        losses['total'].backward()
        optimizer.step()
        scheduler.step()

        if epoch % 200 == 0:
            model.eval()
            x_t = torch.tensor(x_eval, dtype=torch.float32,
                               device=device).view(-1, 1)
            with torch.no_grad():
                p_pred = model(x_t).cpu().numpy().flatten()
            model.train()

            mid = len(x_eval) // 2
            lm = p_pred[:mid].max()
            rm = p_pred[mid:].max()

            # Avoid division by zero
            max_val = max(lm, rm)
            ratio = min(lm, rm) / max_val if max_val > 1e-6 else 0.0

            current_lr = optimizer.param_groups[0]['lr']

            history['epoch'].append(epoch)
            history['left_max'].append(lm)
            history['right_max'].append(rm)
            history['ratio'].append(ratio)
            history['loss_pde'].append(losses['pde'].item())
            history['lr'].append(current_lr)

            if epoch in snapshot_epochs and epoch >= 9000:
                snapshots[epoch] = p_pred.copy()

    return history, snapshots

print("Training seed=0 (full 30000)...")
hist_0, snaps_0 = train_full(seed=0)
print("Training seed=1 (full 30000)...")
hist_1, snaps_1 = train_full(seed=1)

# === Plot 1: Full trajectory of symmetry ===
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for seed, hist, color in [(0, hist_0, 'blue'), (1, hist_1, 'red')]:
    axes[0].plot(hist['epoch'], hist['left_max'],
                 color=color, lw=1.2, label=f'Seed={seed} (Left)')
    axes[0].plot(hist['epoch'], hist['right_max'],
                 color=color, lw=1.2, ls='--', label=f'Seed={seed} (Right)')
    axes[1].plot(hist['epoch'], hist['ratio'],
                 color=color, lw=1.5, label=f'Seed={seed}')
    axes[2].semilogy(hist['epoch'], hist['loss_pde'],
                     color=color, lw=1.2, label=f'Seed={seed}')

# Top Plot: Peak Heights
axes[0].set_ylabel('Peak height')
axes[0].set_title('Left and Right Peak Heights')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Middle Plot: Symmetry Ratio
axes[1].axhline(0.3, color='gray', ls=':', lw=1)
axes[1].axvline(10000, color='orange', ls='--', lw=1.5, label='LR drop x0.1')
axes[1].axvline(20000, color='orange', ls='--', lw=1.5)
axes[1].set_ylabel('Symmetry ratio')
axes[1].set_title('Symmetry Ratio (1 = Perfect, <0.3 = One Peak)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Bottom Plot: PDE Loss
axes[2].axvline(10000, color='orange', ls='--', lw=1.5, label='LR drop x0.1')
axes[2].axvline(20000, color='orange', ls='--', lw=1.5)
axes[2].set_ylabel('PDE Loss')
axes[2].set_xlabel('Epoch')
axes[2].set_title('PDE Loss Convergence')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Full Trajectory: Seed 0 (Success) vs Seed 1 (Failure)', fontsize=14)
plt.tight_layout()
plt.savefig('step6_full_trajectory.png', dpi=100)
plt.show()

# === Table of key epochs ===
key_epochs = [2000, 5000, 9800, 10000, 10200, 15000, 19800, 20000, 20200, 25000, 29800]
print(f"\n{'Epoch':>7} | {'S0 Left':>8} | {'S0 Right':>9} | "
      f"{'S0 Ratio':>9} | {'S0 LR':>8} || "
      f"{'S1 Ratio':>9} | {'S1 LR':>8}")
print("-" * 85)
for e in key_epochs:
    idx = hist_0['epoch'].index(e) if e in hist_0['epoch'] else None
    if idx is None:
        continue
    r0 = hist_0['ratio'][idx]
    r1 = hist_1['ratio'][idx]
    l0 = hist_0['left_max'][idx]
    ri0 = hist_0['right_max'][idx]
    lr0 = hist_0['lr'][idx]
    lr1 = hist_1['lr'][idx]
    print(f"{e:>7} | {l0:>8.4f} | {ri0:>9.4f} | "
          f"{r0:>9.4f} | {lr0:>8.1e} || "
          f"{r1:>9.4f} | {lr1:>8.1e}")

## 3.2.3 Effect of Optimizer Configuration

Compares pure L-BFGS against three Adam + L-BFGS hybrid schedules across 4 seeds. Reproduces Figure 3.4.

In [ ]:

# ─── Experiment setup ────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
problem = DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5)
x_eval  = np.linspace(-2.2, 2.2, 441)
p_exact = problem.exact_solution(x_eval)

SEEDS = [0, 1, 42, 99]

CONFIGS = [
    {
        'label': 'L-BFGS\n5000',
        'optimizer_config': {
            'type':         'lbfgs',
            'epochs':       5000,
            'lr':           1.0,
            'max_iter':     20,
            'history_size': 50,
        },
    },
    {
        'label': 'Adam 1000\n+ L-BFGS 5000\n(before collapse)',
        'optimizer_config': {
            'type':               'hybrid',
            'adam_epochs':      1000,
            'adam_lr':          1e-3,
            'lbfgs_epochs':     5000,
            'lbfgs_lr':         1.0,
            'lbfgs_max_iter':   20,
            'lbfgs_history_size': 50,
        },
    },
    {
        'label': 'Adam 5000\n+ L-BFGS 5000\n(after collapse)',
        'optimizer_config': {
            'type':               'hybrid',
            'adam_epochs':      5000,
            'adam_lr':          1e-3,
            'lbfgs_epochs':     5000,
            'lbfgs_lr':         1.0,
            'lbfgs_max_iter':   20,
            'lbfgs_history_size': 50,
        },
    },
    {
        'label': 'Adam 10000\n+ L-BFGS 5000\n(fine-tuning)',
        'optimizer_config': {
            'type':               'hybrid',
            'adam_epochs':      10000,
            'adam_lr':          1e-3,
            'lbfgs_epochs':     5000,
            'lbfgs_lr':         1.0,
            'lbfgs_max_iter':   20,
            'lbfgs_history_size': 50,
        },
    },
]

# ─── Training runs ───────────────────────────────────────────────────────────

results = {seed: {} for seed in SEEDS}
total_runs = len(SEEDS) * len(CONFIGS)
run = 0

for seed in SEEDS:
    for cfg in CONFIGS:
        run += 1
        label_flat = cfg['label'].replace('\n', ' ')
        print(f"\n[{run}/{total_runs}] Seed={seed} | {label_flat}")

        result = train_model(
            problem,
            model_config={'output_transform': 'softplus'},
            optimizer_config=cfg['optimizer_config'],
            dx=0.01,
            use_normalization=True,
            penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
            seed=seed,
            device=device,
            print_every=99999,  # Print only final line
        )

        model = result['model']
        model.eval()

        x_t = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)
        with torch.no_grad():
            p_pred = model(x_t).cpu().numpy().flatten()

        acc   = acc_L2(p_pred, p_exact)
        mid   = len(x_eval) // 2
        lm    = p_pred[:mid].max()
        rm    = p_pred[mid:].max()

        max_val = max(lm, rm)
        ratio = min(lm, rm) / max_val if max_val > 1e-6 else 0.0
        shape = 'TWO' if ratio > 0.3 else 'ONE'

        results[seed][cfg['label']] = {
            'acc': acc, 'ratio': ratio, 'shape': shape, 'p': p_pred,
        }
        print(f"  -> Acc={acc:.4f} | Ratio={ratio:.3f} | {shape} peak(s)")

# ─── 4x4 Plot ──────────────

fig, axes = plt.subplots(
    len(SEEDS), len(CONFIGS),
    figsize=(5 * len(CONFIGS), 4 * len(SEEDS)),
    sharex=True, sharey=True,
)

for i, seed in enumerate(SEEDS):
    for j, cfg in enumerate(CONFIGS):
        ax  = axes[i, j]
        res = results[seed][cfg['label']]

        ax.plot(x_eval, p_exact, 'k-',  lw=1.5, alpha=0.5, label='Exact')
        ax.plot(x_eval, res['p'], 'r--', lw=1.5,
                label=f"Acc={res['acc']:.3f}")

        # Green border = success (two peaks), red = failure (one peak)
        color = 'green' if res['shape'] == 'TWO' else 'red'
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2.5)

        col_title = cfg['label'].replace('\n', ' | ') if i == 0 else ''
        title_parts = [t for t in [col_title, f"Seed={seed}"] if t]
        ax.set_title('\n'.join(title_parts), fontsize=9)
        ax.legend(fontsize=8, loc='upper left')
        ax.set_xlabel('x', fontsize=9)
        if j == 0:
            ax.set_ylabel(f'Seed={seed}\np(x)', fontsize=10)

plt.suptitle(
    'Experiment 3: Alpha=0.7 | 4 Seeds x 4 Optimizer Configurations\n'
    'Green border = Two peaks (Success), Red border = One peak (Failure)',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig('exp3_optimizer_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Summary Table ───────────────────────────────────────────────────────────

col_labels = [cfg['label'].replace('\n', ' ') for cfg in CONFIGS]
col_w = 35
header = f"{'Seed':>6} | " + " | ".join(f"{c:>{col_w}}" for c in col_labels)
sep = "=" * len(header)

print(f"\n{sep}\n{header}\n{sep}")
for seed in SEEDS:
    row_parts = []
    for cfg in CONFIGS:
        res = results[seed][cfg['label']]
        cell = f"Acc={res['acc']:.3f} | {res['shape']} peak(s)"
        row_parts.append(f"{cell:>{col_w}}")
    print(f"{seed:>6} | " + " | ".join(row_parts))
print(sep)

# Success rate per configuration
print("\nSuccess rate (Ratio > 0.3) per optimizer configuration:")
for cfg in CONFIGS:
    label_flat = cfg['label'].replace('\n', ' ')
    n_success = sum(
        1 for seed in SEEDS
        if results[seed][cfg['label']]['shape'] == 'TWO'
    )
    avg_acc = np.mean([
        results[seed][cfg['label']]['acc'] for seed in SEEDS
    ])
    print(f"  {label_flat:<45} {n_success}/{len(SEEDS)} success "
          f"| Mean Acc={avg_acc:.4f}")

## 3.2.4 Resolving Instability via Transfer Learning

Pre-trains on $\alpha = 0.3$ (reliable convergence) and fine-tunes on $\alpha = 0.7$. Reproduces Figure 3.5.

In [ ]:

x_eval = np.linspace(-2.2, 2.2, 441)

# ─── Step 1: Pre-training on alpha=0.3 ─────

print("=== STEP 1: Pre-training on alpha=0.3 ===")
problem_03 = DoubleWellProblem(alpha=0.3, beta=0.5, sigma=0.5)

result_source = train_model(
    problem=problem_03,
    model_config={'output_transform': 'softplus'},
    optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
    dx=0.01,
    seed=0,
    device=device,
    print_every=10000,
)
pretrained_model = result_source['model']

p_exact_03 = problem_03.exact_solution(x_eval)
pretrained_model.eval()
x_t = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)
with torch.no_grad():
    p_source = pretrained_model(x_t).cpu().numpy().flatten()
acc_source = acc_L2(p_source, p_exact_03)
print(f"Source model acc (alpha=0.3): {acc_source:.4f}")

# ─── Step 2: Baselines — alpha=0.7 without TL (failed seeds) ───

print("\n=== STEP 2: Baselines without Transfer Learning ===")
problem_07 = DoubleWellProblem(alpha=0.7, beta=0.5, sigma=0.5)
p_exact_07 = problem_07.exact_solution(x_eval)

FAILED_SEEDS = [1, 42, 99]
results_baseline = {}

for seed in FAILED_SEEDS:
    print(f"\n  Baseline seed={seed}...")
    result = train_model(
        problem=problem_07,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01,
        seed=seed,
        device=device,
        print_every=99999,
    )
    result['model'].eval()
    with torch.no_grad():
        p_pred = result['model'](x_t).cpu().numpy().flatten()
    acc = acc_L2(p_pred, p_exact_07)
    results_baseline[seed] = {'p': p_pred, 'acc': acc}
    print(f"  -> acc={acc:.4f}")

# ─── Step 3: Transfer Learning — alpha=0.7 with pretrained weights ───────────

print("\n=== STEP 3: Transfer Learning (init from alpha=0.3) ===")
results_tl = {}

for run_idx in range(3):
    print(f"\n  TL run {run_idx+1}/3 (seed={seed})...")
    result = train_model(
        problem=problem_07,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01,
        seed=seed,
        device=device,
        init_model=pretrained_model,
        print_every=99999,
    )
    result['model'].eval()
    with torch.no_grad():
        p_pred = result['model'](x_t).cpu().numpy().flatten()
    acc = acc_L2(p_pred, p_exact_07)
    results_tl[run_idx] = {'p': p_pred, 'acc': acc}
    print(f"  -> acc={acc:.4f}")

# ─── Step 4: Comparison plot ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: baselines (no TL)
ax = axes[0]
ax.plot(x_eval, p_exact_07, 'k-', lw=2, label='Exact solution')
colors = ['#e74c3c', '#e67e22', '#9b59b6']
for (seed, res), color in zip(results_baseline.items(), colors):
    ax.plot(x_eval, res['p'], '--', lw=1.5, color=color,
            label=f'No TL seed={seed} (acc={res["acc"]:.3f})')
ax.set_title('Without Transfer Learning\n(alpha=0.7, Adam 30000)', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: Transfer Learning
ax = axes[1]
ax.plot(x_eval, p_exact_07, 'k-', lw=2, label='Exact solution')
tl_colors = ['#27ae60', '#2ecc71', '#1abc9c']
for (run_idx, res), color in zip(results_tl.items(), tl_colors):
    ax.plot(x_eval, res['p'], '--', lw=1.5, color=color,
            label=f'TL run {run_idx+1} (acc={res["acc"]:.3f})')
ax.set_title('With Transfer Learning\n(pretrained on alpha=0.3)', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Experiment 4: Transfer Learning for alpha=0.7 Double-Well',
             fontsize=13)
plt.tight_layout()
plt.savefig('exp4_transfer_learning.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Summary ───────────

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print("Without TL:")
for seed, res in results_baseline.items():
    mid = len(x_eval) // 2
    lm = res['p'][:mid].max()
    rm = res['p'][mid:].max()

    max_val = max(lm, rm)
    ratio = min(lm, rm) / max_val if max_val > 1e-6 else 0.0

    shape = 'TWO peaks' if ratio > 0.3 else 'ONE peak'
    print(f"  seed={seed}: acc={res['acc']:.4f} | {shape}")

print("With TL:")
for run_idx, res in results_tl.items():
    mid = len(x_eval) // 2
    lm = res['p'][:mid].max()
    rm = res['p'][mid:].max()

    max_val = max(lm, rm)
    ratio = min(lm, rm) / max_val if max_val > 1e-6 else 0.0

    shape = 'TWO peaks' if ratio > 0.3 else 'ONE peak'
    print(f"  run {run_idx+1}: acc={res['acc']:.4f} | {shape}")